# Thesis Note — Milestone 8.1: Natural Missingness Audit

**Project:** Self-Supervised Multimodal Representation Learning for Robust Dental Diagnosis with Naturally Missing Radiographs: A Study on the COde Dataset
**Milestone:** 8.1 — Natural Missingness Audit
**Status:** Completed
**Date:** 2026-08-31

---

## 1. Purpose

The purpose of this sub-milestone was to characterize the naturally occurring missing-modality patterns in the authoritative six-label patient-level benchmark dataset before designing or evaluating any missing-modality method.

This audit was performed to establish:

* how frequently each modality is missing,
* which combinations of available modalities occur naturally,
* how these patterns are distributed across train, validation, and test splits,
* how many labeled visits are available within each pattern,
* and whether the existing patient-level split remains valid.

No samples were removed, no modality was imputed, and the authoritative dataset was not modified.

---

## 2. Authoritative Dataset

The analysis used the finalized six-label patient-level dataset:

```text
results/six_label_patient_level_dataset/labeled_dataset.csv
```

This dataset is derived from the finalized patient-level dataset and preserves the authoritative patient-level split.

### Dataset statistics

| Property            | Value |
| ------------------- | ----: |
| Total visits        | 8,775 |
| Total patients      | 4,800 |
| Labels              |     6 |
| Patient-level split |   Yes |
| Dataset modified    |    No |

The six target labels are:

1. Caries
2. Gingivitis
3. Malocclusion
4. Pulpitis
5. Tooth Loss
6. Tooth Structure Loss

---

## 3. Modality Definition

Three modalities were considered:

* **Image:** intraoral photographs
* **X-ray:** dental radiographs
* **Text:** clinical examination text

A modality was considered missing when no valid representation of that modality was available for the corresponding visit.

For clinical text, empty values such as empty strings, `NaN`, `None`, `null`, `NA`, and `N/A` were treated as missing.

The audit preserved missingness as an observed property of the original data. No imputation was performed.

---

## 4. Natural Modality Availability

The dataset contains the following modality availability:

| Modality | Present | Missing | Missing Rate |
| -------- | ------: | ------: | -----------: |
| Image    |   8,772 |       3 |        0.03% |
| X-ray    |   4,256 |   4,519 |       51.50% |
| Text     |   8,568 |     207 |        2.36% |

The dominant source of missingness is therefore the X-ray modality.

Importantly, X-ray missingness is not a rare edge case: more than half of all visits do not contain a radiograph.

---

## 5. Natural Missing-Modality Patterns

Each visit was assigned a modality-presence pattern using:

```text
111 = Image + X-ray + Text
110 = Image + X-ray
101 = Image + Text
100 = Image only
011 = X-ray + Text
001 = Text only
```

The observed patterns were:

| Code | Modality Pattern     | Visits | Patients | Percentage |
| ---- | -------------------- | -----: | -------: | ---------: |
| 111  | Image + X-ray + Text |  4,137 |    2,803 |     47.15% |
| 110  | Image + X-ray        |    118 |      117 |      1.34% |
| 101  | Image + Text         |  4,428 |    2,817 |     50.46% |
| 100  | Image only           |     89 |       85 |      1.01% |
| 011  | X-ray + Text         |      1 |        1 |      0.01% |
| 001  | Text only            |      2 |        1 |      0.02% |

There were therefore six naturally occurring modality patterns.

---

## 6. Mapping to Missing-Modality Scenarios

The four predefined experimental scenarios can be mapped to the naturally occurring patterns as follows:

| Scenario          | Available Modalities | Natural Pattern |
| ----------------- | -------------------- | --------------- |
| A — Complete      | Image + X-ray + Text | 111             |
| B — X-ray Missing | Image + Text         | 101             |
| C — Image Only    | Image                | 100             |
| D — Text Missing  | Image + X-ray        | 110             |

The remaining patterns (`011` and `001`) occur extremely rarely and are therefore treated as **other natural patterns**, rather than primary experimental scenarios.

This distinction prevents the main experiments from being dominated by extremely small populations.

---

## 7. Label Coverage Within Missingness Patterns

The number of visits containing at least one of the six target labels was also examined.

| Pattern              | Visits | Labeled Visits | Label Coverage |
| -------------------- | -----: | -------------: | -------------: |
| Image + X-ray + Text |  4,137 |          3,227 |         78.00% |
| Image + X-ray        |    118 |              1 |          0.85% |
| Image + Text         |  4,428 |          3,407 |         76.94% |
| Image only           |     89 |              3 |          3.37% |
| X-ray + Text         |      1 |              1 |        100.00% |
| Text only            |      2 |              2 |        100.00% |

The primary complete and X-ray-missing patterns contain substantial numbers of labeled visits.

However, the Image + X-ray and Image-only patterns have extremely low six-label coverage. Therefore, these patterns should be interpreted carefully in downstream performance evaluation.

In particular, their small labeled populations make them unsuitable for strong standalone conclusions without additional controlled experiments.

---

## 8. Distribution Across Patient-Level Splits

The natural missingness patterns were also inspected separately within the authoritative train, validation, and test splits.

### Train

| Pattern       | Visits |
| ------------- | -----: |
| Complete      |  2,896 |
| X-ray Missing |  3,097 |
| Text Missing  |     75 |
| Image Only    |     58 |
| X-ray + Text  |      1 |
| Text Only     |      2 |

Total train visits: **6,129**

### Validation

| Pattern       | Visits |
| ------------- | -----: |
| Complete      |    616 |
| X-ray Missing |    672 |
| Text Missing  |     26 |
| Image Only    |     16 |

Total validation visits: **1,330**

### Test

| Pattern       | Visits |
| ------------- | -----: |
| Complete      |    625 |
| X-ray Missing |    659 |
| Text Missing  |     17 |
| Image Only    |     15 |

Total test visits: **1,316**

The dominant two patterns remain consistent across all three splits:

* Complete multimodal: approximately 46–47%
* Image + Text with X-ray missing: approximately 50%

This is important because it shows that X-ray missingness is not concentrated exclusively in one split.

---

## 9. Patient-Level Split Validation

The audit verified that the existing patient-level split remains valid.

Validation results:

```text
patient_level_split: PASS
visit_uniqueness: PASS
authoritative_six_label_dataset: PASS
```

The audit did not regenerate or modify the train/validation/test assignment.

Therefore, all subsequent Milestone 8 experiments must continue to use the existing patient-level split.

---

## 10. Important Findings

The audit establishes several important facts for the next experiments.

### 10.1 X-ray missingness is the dominant missing-modality problem

There are:

```text
4,519 visits with missing X-ray
```

representing:

```text
51.50% of all visits
```

Therefore, missing X-ray is the most important naturally occurring missing-modality condition in the COde dataset.

### 10.2 The X-ray-missing scenario has a large evaluation population

The `Image + Text` pattern contains:

```text
4,428 visits
```

and is therefore large enough to support meaningful evaluation.

This makes Scenario B — X-ray Missing — the primary natural missing-modality scenario.

### 10.3 Complete cases are only a subset of the dataset

Only:

```text
4,137 / 8,775 = 47.15%
```

of visits contain all three modalities.

This provides an important motivation for investigating missing-modality robustness: a complete-case multimodal approach cannot directly use more than half of the visits.

### 10.4 Rare missingness patterns require caution

Missing text and missing image cases exist naturally, but their labeled populations are much smaller.

Therefore, performance on these cases should not be used alone to make strong conclusions.

For these scenarios, controlled missingness experiments will be important to obtain sufficiently large and balanced evaluation populations.

---

## 11. Methodological Decision for Milestone 8

The results support a two-part evaluation strategy for missing-modality robustness.

### Part 1 — Natural Missingness

Use the naturally occurring missingness in the COde dataset, particularly:

```text
Complete:
Image + X-ray + Text

X-ray Missing:
Image + Text
```

This measures robustness under the real missingness distribution of the dataset.

### Part 2 — Controlled Missingness

Artificially mask modalities in a controlled evaluation setting while preserving the same underlying samples.

This will allow fair comparison between modality conditions when natural missingness produces very different sample sizes.

Controlled missingness must not alter the authoritative dataset or patient-level split.

---

## 12. What This Sub-Milestone Does Not Do

Milestone 8.1 is an **audit only**.

It does not:

* train a new model,
* modify the SSL encoders,
* modify the Fusion architecture,
* impute missing modalities,
* remove incomplete samples,
* regenerate the patient-level split,
* compare model performance,
* or claim robustness.

The purpose is to establish the experimental population and missingness distribution before model evaluation.

---

## 13. Outputs

The following artifacts were generated:

```text
results/milestone8_missing_modality/
└── 01_natural_missingness/
    ├── natural_missingness_summary.json
    └── modality_patterns_by_split.csv
```

The implementation is located under:

```text
src/missing_modality/
```

The authoritative six-label dataset remains:

```text
results/six_label_patient_level_dataset/labeled_dataset.csv
```

---

## 14. Conclusion

Milestone 8.1 confirms that modality missingness is a substantial property of the COde dataset rather than an artificially constructed problem.

The most important observation is that **51.50% of visits are naturally missing X-ray data**, while the complete multimodal population represents only **47.15%** of visits.

Consequently, the next stage of Milestone 8 will evaluate whether the existing SSL-based multimodal fusion model can maintain diagnostic performance when one or more modalities are unavailable.

The central question for the next experiment is therefore not whether missingness exists, but:

> **How much performance does the existing multimodal model lose when a modality is unavailable, and can a missing-modality-aware fusion strategy reduce this degradation?**

---

**Milestone 8.1 Status: COMPLETED**

**Next:** Milestone 8.2 — Missing-Modality Baseline Evaluation


# Milestone 8.2 — Controlled Missing-Modality Robustness Evaluation

## 3. Model Architecture

The existing **SSL Fusion — Main** model uses the following architecture:

```text
Photograph representation: 2048 → 512
Radiograph representation: 2048 → 512
Text representation:       768  → 512

                    ↓

              Concatenation

                    ↓

                  1536

                    ↓

              Fusion MLP

                    ↓

                   512

                    ↓

             6-label output
```

### Checkpoint

```text
results/fusion/main/best_model.pt
```

The model was trained previously during **Milestone 7.3**.

### Important Protocol Constraint

No training or fine-tuning was performed during this experiment.

Therefore:

* The model architecture was unchanged.
* The model parameters were unchanged.
* The existing checkpoint was used directly.
* The test set was not used for model selection.

This ensures that the experiment measures the **robustness of the existing fusion model** rather than the effect of additional training.

---

## 4. Controlled Test Population

The evaluation uses the **same complete-case test population** used by the original Milestone 7.3 Fusion experiment.

**Number of test samples:**

```text
633
```

Every sample in this population originally contains:

```text
Photograph + Radiograph + Clinical Text
```

The same 633 samples were evaluated under every scenario.

This is important because it ensures that differences between scenarios are caused by the **missing-modality condition** rather than by changes in the evaluated population.

---

## 5. Controlled Missing-Modality Protocol

Missing modalities were simulated at the **representation level**.

For a missing modality, its SSL representation was replaced by a **zero vector with the same dimensionality** as the original representation.

No data imputation was performed.

### Example

**Complete:**

```text
Image representation
        +
X-ray representation
        +
Text representation
```

**Missing X-ray:**

```text
Image representation
        +
Zero X-ray representation
        +
Text representation
```

This allows the existing fusion model to be evaluated under identical samples while systematically removing modality information.

---

## 6. Evaluation Scenarios

Four controlled scenarios were evaluated.

### Scenario A — Complete

**Input:**

```text
Image + X-ray + Text
```

**Missing modalities:**

```text
None
```

This is the reference condition.

---

### Scenario B — Missing X-ray

**Input:**

```text
Image + Text
```

**Missing modality:**

```text
X-ray
```

The radiograph representation was replaced with a zero vector.

---

### Scenario C — Missing Text

**Input:**

```text
Image + X-ray
```

**Missing modality:**

```text
Clinical Text
```

The text representation was replaced with a zero vector.

---

### Scenario D — Missing Multiple Modalities

**Input:**

```text
Image Only
```

**Missing modalities:**

```text
X-ray + Text
```

Both corresponding representations were replaced with zero vectors.

---

## 7. Results

The results were:

| Scenario                 | Input Modalities     | Samples | Macro F1 | Micro F1 |  AUROC | Accuracy |
| ------------------------ | -------------------- | ------: | -------: | -------: | -----: | -------: |
| A — Complete             | Image + X-ray + Text |     633 |   0.7676 |   0.8760 | 0.9646 |   0.8120 |
| B — Missing X-ray        | Image + Text         |     633 |   0.6751 |   0.8260 | 0.9609 |   0.7378 |
| C — Missing Text         | Image + X-ray        |     633 |   0.3559 |   0.5366 | 0.7508 |   0.4850 |
| D — Missing X-ray + Text | Image Only           |     633 |   0.2441 |   0.2606 | 0.7200 |   0.2006 |

---

## 8. Performance Degradation

Performance degradation was calculated relative to the complete-case condition.

### Macro F1

| Scenario             | Macro F1 |   Drop |
| -------------------- | -------: | -----: |
| Complete             |   0.7676 | 0.0000 |
| Missing X-ray        |   0.6751 | 0.0925 |
| Missing Text         |   0.3559 | 0.4118 |
| Missing X-ray + Text |   0.2441 | 0.5236 |

The largest degradation occurs when **Text is unavailable**.

Removing X-ray produces a substantially smaller degradation.

Removing both X-ray and Text produces the largest overall performance loss.

---

### Micro F1

| Scenario             | Micro F1 |   Drop |
| -------------------- | -------: | -----: |
| Complete             |   0.8760 | 0.0000 |
| Missing X-ray        |   0.8260 | 0.0501 |
| Missing Text         |   0.5366 | 0.3395 |
| Missing X-ray + Text |   0.2606 | 0.6155 |

The same pattern is observed for Micro F1.

---

### AUROC

| Scenario             |  AUROC |   Drop |
| -------------------- | -----: | -----: |
| Complete             | 0.9646 | 0.0000 |
| Missing X-ray        | 0.9609 | 0.0037 |
| Missing Text         | 0.7508 | 0.2138 |
| Missing X-ray + Text | 0.7200 | 0.2447 |

Interestingly, missing X-ray causes only a **very small AUROC reduction**, despite the decrease in F1.

---

## 9. Interpretation

The controlled experiment demonstrates that the existing **SSL Fusion — Main** model is **not robust to missing modalities**.

The degradation is highly modality-dependent.

### Missing X-ray

When the radiograph representation is removed:

```text
Macro F1:
0.7676 → 0.6751

Drop:
0.0925
```

The model retains relatively strong discriminative ability.

This is consistent with the earlier dataset audit showing that radiographs are the naturally scarce modality in COde.

---

### Missing Text

When clinical text is removed:

```text
Macro F1:
0.7676 → 0.3559

Drop:
0.4118
```

This represents a substantial degradation.

This finding is also consistent with the previous SSL downstream experiments, where the text modality produced the strongest single-modality performance.

Therefore, the current fusion model appears to depend strongly on the information contained in the **clinical text representation**.

---

### Multiple Missing Modalities

When both X-ray and Text are removed:

```text
Macro F1:
0.7676 → 0.2441

Drop:
0.5236
```

The model performs poorly when only the image representation remains.

This demonstrates that the current fusion architecture cannot simply be expected to remain robust when its inputs become incomplete.

---

## 10. Main Finding

The most important finding of **Milestone 8.2** is:

> **The existing SSL Fusion — Main model experiences substantial performance degradation under missing-modality conditions, particularly when clinical text is unavailable.**

Therefore, simply applying the existing complete-case fusion model to incomplete multimodal inputs is insufficient.

This provides direct experimental motivation for the next stage of the research.

---

## 11. Relationship to Natural Missingness

The previous **Milestone 8.1** audit showed that natural missingness is substantial in the COde dataset.

The most important naturally occurring patterns were:

| Modality Pattern     | Visits | Percentage |
| -------------------- | -----: | ---------: |
| Image + X-ray + Text |  4,137 |     47.15% |
| Image + Text         |  4,428 |     50.46% |
| Image + X-ray        |    118 |      1.34% |
| Image Only           |     89 |      1.01% |

Therefore, the controlled experiment is particularly relevant to the real dataset.

The most common natural pattern is:

```text
Image + Text
X-ray Missing
```

which corresponds directly to **Scenario B**.

However, the controlled experiment is intentionally performed on the **fixed complete-case test population** so that the effect of missingness can be isolated without confounding the comparison by changing the evaluated samples.

Natural missingness will be evaluated separately in a later sub-milestone.

---

## 12. Research Implication

Milestone 8.2 establishes the **baseline failure mode** that the proposed missing-modality method must address.

The goal of the next stage is not necessarily to obtain a higher complete-case Macro F1 than the existing Main Fusion model.

Instead, the primary objective is:

> **Reduce performance degradation when one or more modalities are missing.**

A successful robust model should therefore preserve substantially more of its performance across:

```text
Complete
      ↓
Missing X-ray
      ↓
Missing Text
      ↓
Missing Multiple Modalities
```

while maintaining a comparable complete-case performance.

---

## 13. Next Step — Milestone 8.3

The next stage will design and train a **Missing-Modality Robust Fusion** model.

Unlike Milestone 8.2, this stage will involve model training.

The training protocol will use the **authoritative six-label patient-level dataset** and its existing patient-level split.

The training process will expose the model to different modality-availability patterns so that it learns to operate when one or more modalities are unavailable.

The resulting model will then be evaluated under the same controlled scenarios used here.

The primary comparison will therefore be:

```text
Existing Main Fusion
        vs
Missing-Modality Robust Fusion
```

with particular attention to:

```text
Performance under missing modalities
        and
Performance degradation from Complete
```

---

## 14. Validation and Reproducibility

The following protocol checks passed:

| Check                                     | Status                     |
| ----------------------------------------- | -------------------------- |
| Complete-case test population             | PASS                       |
| Same population across all scenarios      | PASS                       |
| Patient-level split                       | Inherited from Milestone 7 |
| Test set used for model selection         | NO                         |
| Model modified                            | NO                         |
| Model retrained                           | NO                         |
| Fine-tuning performed                     | NO                         |
| Imputation performed                      | NO                         |
| Zero-vector replacement used consistently | YES                        |

---

## 15. Conclusion

Milestone 8.2 successfully established a **controlled robustness baseline** for the existing SSL Fusion — Main model.

The complete-case performance was:

```text
Macro F1 = 0.7676
Micro F1 = 0.8760
AUROC    = 0.9646
Accuracy = 0.8120
```

Under missing X-ray conditions, Macro F1 decreased by only:

```text
0.0925
```

whereas missing Text caused a much larger decrease of:

```text
0.4118
```

and missing both X-ray and Text caused a decrease of:

```text
0.5236
```

These results demonstrate a clear robustness limitation in the existing fusion approach.

Consequently, the next stage will focus on designing a fusion model that **explicitly learns to handle missing modalities** rather than assuming that all modalities are always available.
